In [1]:
import pandas as pd
import re

In [23]:
base_path_data = "../../../data"
base_path_cv = base_path_data + "/processed/model_performance/cross_validation"
base_path_held_out = base_path_data + "/processed/model_performance/held_out"

# read metadata
metadata = pd.read_json(f"{base_path_data}/raw/metadata_ku.json", lines=True)
metadata_unique = metadata.groupby("new_sub_id").first().reset_index()

## Cross validation model performance

In [59]:
# function to extract subject ID from evaluation results filename
def extract_sub_id(filename):
    match = re.match(r"colon_(0*\d+)-", filename)
    if match:
        return f"sub{int(match.group(1)):03d}"
    return None

# read results
cv_results = {}
for fold_number in range(5):
    df = pd.read_csv(f"{base_path_cv}/results_fold_{fold_number}.csv")
    # group by filename and keep only the first row per file as we look at overall performance
    df = df.groupby("filename").first().reset_index()
    df["sub_id"] = df["filename"].apply(extract_sub_id)

    # merge evaluation results with metadata on subject ID (per scan)
    df = pd.merge(
        df,
        metadata_unique,
        left_on="sub_id",
        right_on="new_sub_id",
        how="left"
    )

    df["gender"] = df["gender"].apply(lambda x: x if x in ["M", "F"] else "U")

    metrics = {
        "hf95": {
            "count": df.groupby("gender")["overall_dice_score"].count(),
            "mean": df.groupby("gender")["overall_dice_score"].mean(),
            "std": df.groupby("gender")["overall_dice_score"].std()
        }
    }
        
    cv_results[fold_number] = metrics

In [ ]:
cv_results[1]
# todo create plot and print all results nicely

{'hf95': {'count': gender
  F    66
  M    56
  U    15
  Name: overall_dice_score, dtype: int64,
  'mean': gender
  F    0.973294
  M    0.970655
  U    0.963584
  Name: overall_dice_score, dtype: float64,
  'std': gender
  F    0.011489
  M    0.013785
  U    0.026107
  Name: overall_dice_score, dtype: float64}}